In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
# # Install explainability libraries
# !pip install shap lime
import shap
import lime
import lime.lime_tabular



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/547.2 kB ? eta -:--:--
   ------------------- -------------------- 262.1/547.2 kB ? eta -:--:--
   ---------------------------------------- 547.2/547.2 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 1.3 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.7 MB 1.7 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.7 MB 1.6 MB/s eta 0:00:01
   --------------------------- ------------ 1.8/2.7 MB 1.8 MB/s eta 0:00:01
   ----------------------------------- ---- 2.4/2.7 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 1.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
    ------

c:\Users\gchehata\scoop\apps\miniconda3\4.12.0\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv('../AirlineScrappedReview_Cleaned_seif_we_ibra#1.csv')

In [ ]:




X = df[['Flying_Date', 'Route', 'Verified', 'Review_title', 'Review_content', 'Traveller_Type', 'Class', 'Start_Location', 'End_Location', 'Layover_Route', 'Start_Longitude', 'Start_Latitude', 'End_Longitude', 'End_Latitude', 'Start_Address', 'End_Address']]

y = df['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)



In [ ]:
# Step 1: Feature Engineering - Handle Categorical & Numerical Features
print("="*80)
print("FEATURE ENGINEERING FOR CLASSICAL ML")
print("="*80)

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical features ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Handle missing values
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Fill numerical missing values with median
for col in numerical_cols:
    median_val = X_train_clean[col].median()
    X_train_clean[col].fillna(median_val, inplace=True)
    X_test_clean[col].fillna(median_val, inplace=True)

# Fill categorical missing values with 'Unknown'
for col in categorical_cols:
    X_train_clean[col].fillna('Unknown', inplace=True)
    X_test_clean[col].fillna('Unknown', inplace=True)

print(f"\n✓ Missing values handled")

# One-Hot Encode categorical features
X_train_encoded = pd.get_dummies(X_train_clean, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_clean, columns=categorical_cols, drop_first=True)

# Ensure both sets have the same columns
missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0

X_test_encoded = X_test_encoded[X_train_encoded.columns]

print(f"✓ One-hot encoding applied")
print(f"  Total features after encoding: {X_train_encoded.shape[1]}")
print(f"  X_train shape: {X_train_encoded.shape}")
print(f"  X_test shape: {X_test_encoded.shape}")


In [ ]:
# Step 2: Train Classical ML Model (RandomForest)
print("\n" + "="*80)
print("TRAINING CLASSICAL ML MODEL")
print("="*80)

# Train RandomForest Classifier
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

print("\nTraining RandomForestClassifier...")
model.fit(X_train_encoded, y_train)

# Evaluate model
y_pred = model.predict(X_test_encoded)
accuracy = accuracy_score(y_test, y_pred)

print(f"✓ Model trained successfully")
print(f"  Test Accuracy: {accuracy:.4f}")

# Get feature names for SHAP
feature_names = X_train_encoded.columns.tolist()
print(f"  Number of features: {len(feature_names)}")


In [ ]:
# Step 3: Initialize SHAP Explainer (TreeExplainer for RandomForest)
print("\n" + "="*80)
print("SHAP EXPLAINER INITIALIZATION")
print("="*80)

print("\nInitializing SHAP TreeExplainer (optimized for tree-based models)...")

# TreeExplainer is the most efficient explainer for tree-based models
explainer = shap.TreeExplainer(model)

print("✓ TreeExplainer created successfully")
print(f"  Explainer type: {type(explainer).__name__}")
print(f"  Model type: {type(model).__name__}")

# Calculate SHAP values for test set (this may take a moment)
print("\nCalculating SHAP values for test set...")
print("(This computes feature contributions for each prediction)")

shap_values = explainer.shap_values(X_test_encoded)

print(f"✓ SHAP values calculated")
print(f"  SHAP values shape: {np.array(shap_values).shape if isinstance(shap_values, list) else shap_values.shape}")
print(f"  Explainer base_value (average model output): {explainer.expected_value}")

# For binary/multiclass classification, handle SHAP values appropriately
if isinstance(shap_values, list):
    print(f"  Multi-class classification detected: {len(shap_values)} classes")
    shap_values_to_use = shap_values[1]  # Use class 1 for binary, or adjust for multiclass
else:
    shap_values_to_use = shap_values


In [ ]:
# Step 4: SHAP Summary Plot (Global Feature Importance)
print("\n" + "="*80)
print("SHAP SUMMARY PLOT - GLOBAL FEATURE IMPORTANCE")
print("="*80)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Plot 1: Bar plot - Mean absolute SHAP values (which features matter most?)
print("\n1. Creating Bar Plot (top 15 most important features)...")
plt.sca(axes[0])
shap.summary_plot(shap_values_to_use, X_test_encoded, plot_type="bar", max_display=15, show=False)
axes[0].set_title("SHAP Bar Plot: Top 15 Most Important Features\n(Average absolute impact on model output)", fontsize=12, fontweight='bold')

# Plot 2: Bee swarm plot - Feature values vs SHAP values (how features affect predictions?)
print("2. Creating Bee Swarm Plot (feature impact distribution)...")
plt.sca(axes[1])
shap.summary_plot(shap_values_to_use, X_test_encoded, plot_type="dot", max_display=15, show=False)
axes[1].set_title("SHAP Bee Swarm Plot: Top 15 Features\n(Red=High feature value, Blue=Low feature value)", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('shap_summary_plots.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Summary plots created and saved as 'shap_summary_plots.png'")
print("\nInterpretation:")
print("  • Bar Plot: Shows which features have the largest average impact on predictions")
print("  • Bee Swarm: Shows if high/low feature values push predictions up or down")
print("    - Red dots: High feature values")
print("    - Blue dots: Low feature values")
print("    - Position on x-axis: Positive = increases prediction, Negative = decreases prediction")


In [ ]:
# Step 5: SHAP Force Plots (Individual Prediction Explanations)
print("\n" + "="*80)
print("SHAP FORCE PLOTS - INDIVIDUAL PREDICTION EXPLANATIONS")
print("="*80)

# Select a few interesting samples to explain
sample_indices = [0, 10, 20]  # You can change these

for idx, sample_idx in enumerate(sample_indices):
    print(f"\n--- Sample {idx+1}: Test Index {sample_idx} ---")
    print(f"Predicted Rating: {model.predict(X_test_encoded.iloc[[sample_idx]])[0]}")
    print(f"Actual Rating: {y_test.iloc[sample_idx]}")
    
    # Create force plot
    shap.force_plot(
        explainer.expected_value,
        shap_values_to_use[sample_idx],
        X_test_encoded.iloc[sample_idx],
        feature_names=feature_names,
        matplotlib=True,
        show=False
    )
    plt.tight_layout()
    plt.savefig(f'shap_force_plot_sample_{sample_idx}.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Force plot saved as 'shap_force_plot_sample_{sample_idx}.png'")

print("\n\nInterpretation of Force Plots:")
print("  • Base value (gray): Average prediction across dataset")
print("  • Red arrows (→): Features pushing prediction UP (positive contribution)")
print("  • Blue arrows (←): Features pushing prediction DOWN (negative contribution)")
print("  • Arrow length: Magnitude of feature's contribution")
print("  • Output value: Final prediction")


In [ ]:
# Step 6: SHAP Dependence Plots (Feature Interactions)
print("\n" + "="*80)
print("SHAP DEPENDENCE PLOTS - FEATURE INTERACTIONS & RELATIONSHIPS")
print("="*80)

# Get top features by mean absolute SHAP value
mean_abs_shap = np.abs(shap_values_to_use).mean(axis=0)
top_features_idx = np.argsort(mean_abs_shap)[-4:][::-1]  # Top 4 features
top_features_names = [feature_names[i] for i in top_features_idx]

print(f"\nAnalyzing top 4 most important features:")
for feat_name in top_features_names:
    print(f"  • {feat_name}")

# Create dependence plots for top features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, feat_idx in enumerate(top_features_idx):
    plt.sca(axes[i])
    print(f"\nCreating dependence plot for: {feature_names[feat_idx]}")
    
    shap.dependence_plot(
        feat_idx,
        shap_values_to_use,
        X_test_encoded,
        feature_names=feature_names,
        show=False,
        ax=axes[i]
    )
    axes[i].set_title(f"Dependence: {feature_names[feat_idx]}\n(Feature value vs SHAP contribution)", 
                      fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('shap_dependence_plots.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Dependence plots created and saved as 'shap_dependence_plots.png'")
print("\nInterpretation:")
print("  • X-axis: Actual feature value")
print("  • Y-axis: SHAP contribution (impact on prediction)")
print("  • Scatter points: Each test sample")
print("  • Blue/Red coloring: Interaction with another feature (auto-detected by SHAP)")
print("  • Pattern: Shows if feature has non-linear or interactive effects")


In [ ]:
# Step 7: SHAP Waterfall Plots (Individual Prediction Decomposition)
print("\n" + "="*80)
print("SHAP WATERFALL PLOTS - DETAILED PREDICTION BREAKDOWN")
print("="*80)

# Create explanation objects for waterfall plots
from shap import Explanation

for sample_idx in [0, 5, 15]:
    print(f"\n--- Waterfall Plot for Test Sample {sample_idx} ---")
    print(f"Predicted Rating: {model.predict(X_test_encoded.iloc[[sample_idx]])[0]}")
    print(f"Actual Rating: {y_test.iloc[sample_idx]}")
    
    # Create explanation object
    explanation = Explanation(
        values=shap_values_to_use[sample_idx],
        base_values=explainer.expected_value,
        data=X_test_encoded.iloc[sample_idx],
        feature_names=feature_names
    )
    
    # Create waterfall plot (shows top 15 features)
    plt.figure(figsize=(12, 8))
    shap.waterfall_plot(explanation, max_display=15, show=False)
    plt.title(f"SHAP Waterfall: Test Sample {sample_idx}\n(How features contribute to final prediction)", 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'shap_waterfall_sample_{sample_idx}.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Waterfall plot saved as 'shap_waterfall_sample_{sample_idx}.png'")

print("\n\nInterpretation of Waterfall Plots:")
print("  • Bottom gray bar: Base value (average prediction)")
print("  • Red bars (→): Features increasing prediction")
print("  • Blue bars (←): Features decreasing prediction")
print("  • Top value: Final model prediction")
print("  • Shows top 15 features by absolute contribution")
print("  • Arrows indicate direction and magnitude of each feature's impact")


In [ ]:
# Step 8: SHAP Heatmap (Multiple Predictions Overview)
print("\n" + "="*80)
print("SHAP HEATMAP - MULTI-INSTANCE OVERVIEW")
print("="*80)

# Create a heatmap showing SHAP values for multiple instances
plt.figure(figsize=(14, 8))
print("Creating SHAP heatmap for first 50 test samples...")
shap.summary_plot(shap_values_to_use[:50], X_test_encoded.iloc[:50], plot_type="dot", 
                  max_display=15, show=False)
plt.title("SHAP Heatmap: Feature Contributions for 50 Test Samples\n(Allows pattern recognition across multiple predictions)", 
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ SHAP heatmap created and saved as 'shap_heatmap.png'")


## Summary: SHAP Explainability for Classical ML Models

### What We've Created:

1. **Summary Plot (Bar + Bee Swarm)**
   - Global feature importance ranking
   - Shows which features matter most on average
   - Shows direction of impact (positive/negative)

2. **Force Plots**
   - Individual prediction explanations
   - Visualizes base value → final prediction
   - Shows which features pushed the prediction up or down

3. **Dependence Plots**
   - Feature relationships and interactions
   - Shows if features have non-linear effects
   - Helps identify feature dependencies

4. **Waterfall Plots**
   - Detailed breakdown of single predictions
   - Shows exact contribution of each feature
   - Useful for understanding specific model decisions

5. **Heatmap**
   - Multi-instance overview
   - Helps identify patterns across many predictions
   - Good for finding systematic biases

### Key Insights for Your Model:

- **Top Features**: See summary plot for most influential features
- **Linear vs Non-linear**: Dependence plots show if effects are simple or complex
- **Feature Interactions**: SHAP values automatically capture interactions
- **Prediction Reliability**: Force plots help identify if logic seems reasonable

### When to Use Each Plot:

| Plot Type | Best For | Use Case |
|-----------|----------|----------|
| **Summary (Bar)** | Quick importance overview | Stakeholder presentations |
| **Summary (Bee Swarm)** | Direction & distribution | Understanding feature impacts |
| **Force Plot** | Single prediction explanation | Debugging specific decisions |
| **Dependence** | Feature relationships | Understanding model logic |
| **Waterfall** | Detailed breakdown | Detailed analysis of key predictions |
| **Heatmap** | Pattern recognition | Identifying systematic effects |

### Advantages of SHAP over other methods:

✓ **Theoretically sound**: Based on Shapley values from game theory  
✓ **Model-agnostic options**: TreeExplainer for RF, KernelExplainer for any model  
✓ **Fast for trees**: TreeExplainer is computationally efficient for RandomForest  
✓ **Captures interactions**: Automatically includes feature interaction effects  
✓ **Consistent**: Satisfies desirable properties (local accuracy, consistency)  
